# Useful Notebook: Generate Custom Composite Feature Importance Plots
**This notebook will allow users to generate custom variations of the composite feature importance plots.**

*This notebook is designed to run after having run STREAMLINE (at least phases 1-6) and will use the files from a specific STREAMLINE experiment folder, as well as save new output files to that same folder.*

***
## Notebook Details
Generates custom feature importance plots: (1) to include all features in the dataset or (2) some different number of top features, (3) composite FI plots that also includes fractionation (each algorithm has limited total bar area to fill in the overall plot), or (4) to provide a way to generate modified versions of these plots without rerunning or directly editing the code in the original pipeline. 

When run, 'as-is' this notebook will generate 4 feature composite plots: (1) normalized only, (2) normalized and weighted, (3) normalized and fractionated, and (4) normalized, weighted, and fractionated. However these plots will illustrate all features in the processed data (unless the user changes `top_model_features`).  These plots will also present mean FI scores (but the user can change this to median using `fi_ranking`, and they will weight these FI scores using mean model balanced accuracy (however users can change this to median, and some other metric weighting with `fi_weighting` and `metric_weight`, respectively. These will be saved in the same location as the original composite feature importance plots within the experiment folder.

***
## Notebook Run Parameters
* This notbook has been set up to run 'as-is' on the experiment folder generated when running the demo of STREAMLINE in any mode (if no run parameters were changed). 
* If you have run STREAMLINE on different target data or saved the experiment to some other folder outside of STREAMLINE, you need to edit `experiment_path` below to point to the respective experiment folder.

In [ ]:
experiment_path = "/Users/harshbandhey/Local/Cedars/Urbslab/STREAMLINEv3New/test/out_full_pipeline/DemoExp" # path the target experiment folder 
targetDataName = None # 'None' if user wants to generate visualizations for all analyzed datasets
algorithms = [] # use empty list if user wishes to plot feature importance for all modeling algorithms that were run in pipeline.
top_model_features = None # None - to plot all features in original dataset, or specify some (int) to indicate # of top features to plot.
name_modifier = '_AllFeatures' # Modifies standard composite FI plot filename to avoid overwriting originals.
viz_norm_only = True # Generate plot with only FI normalization
viz_norm_weight = True #Generate plot with FI normalization and performance metric weighting
viz_norm_frac = True # Generate plot with FI normalization and fractionation (each algorithm has limited bar area to distribute in plot)
viz_norm_weight_frac = True #Generate plot with FI normalization performance metric weighting and fractionation
legend_inside_plot = True # place legend ouside plot in upper right hand corner, other wise placed inside on upper right hand corner.
fi_ranking = 'mean' # specify either 'mean' or 'median' to indicate whether to take the mean or median CV FI value for these plots.
fi_weighting = 'mean' #sepcify either 'mean' or 'median' to indicate whether to take the mean or median of selected evaluation metric to weigh FI scores in composite FI plot
metric_weight = 'Balanced Accuracy' # model evaluation metric used to weigh model feature importances

***
## Housekeeping
### Import Packages

In [ ]:
import os
import pandas as pd
import pickle
from statistics import mean, stdev, median
import matplotlib.pyplot as plt
from matplotlib import rc
import numpy as np
#from streamline.modeling.utils import ABBREVIATION, COLORS
import seaborn as sns
sns.set_theme()

import warnings
warnings.filterwarnings('ignore')

# Jupyter Notebook Hack: This code ensures that the results of multiple commands within a given cell are all displayed, rather than just the last. 
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## Automatically detect data folder names

In [ ]:
# Get dataset paths for all completed dataset analyses in experiment folder
experiment_name = experiment_path.split('/')[-1]
remove_list = {
    '.DS_Store', 'metadata.pickle', 'metadata.csv', 'algInfo.pickle',
    'DatasetComparisons', 'jobs', 'jobsCompleted', 'logs', 'KeyFileCopy', 'dask_logs',
    'reporting', 'reporting_replication', 'run_params.pickle', 'runtime',
    experiment_name + '_STREAMLINE_Report.pdf'
}

datasets = []
for d in sorted(os.listdir(experiment_path)):
    dpath = os.path.join(experiment_path, d)
    if d in remove_list or not os.path.isdir(dpath):
        continue
    has_exploratory = os.path.isdir(os.path.join(dpath, 'exploratory'))
    has_model_data = os.path.isdir(os.path.join(dpath, 'model_evaluation')) or os.path.isdir(os.path.join(dpath, 'models'))
    if has_exploratory and has_model_data:
        datasets.append(d)

print("Analyzed Datasets: " + str(datasets))

## Load other necessary parameters

In [ ]:
# Unpickle metadata from previous phase
file = open(experiment_path+'/'+"metadata.pickle", 'rb')
metadata = pickle.load(file)
file.close()
# Load variables specified earlier in the pipeline from metadata
outcome_label = metadata.get('Outcome Label', metadata.get('Class Label', 'Class'))
instance_label = metadata['Instance Label']
cv_partitions = int(metadata['CV Partitions'])

requested_algorithms = list(algorithms) if isinstance(algorithms, list) else []

# Unpickle algorithm information from previous phase
alg_info_path = os.path.join(experiment_path, "algInfo.pickle")
if os.path.exists(alg_info_path):
    with open(alg_info_path, "rb") as file:
        algInfo = pickle.load(file)
else:
    algInfo = {}

algorithms = []
abbrev = {}
colors = {}
algColors = []
for key, value in algInfo.items():
    if isinstance(value, (list, tuple)) and len(value) > 0 and bool(value[0]):
        algorithms.append(key)
        abbrev[key] = value[1] if len(value) > 1 else key
        colors[key] = value[2] if len(value) > 2 else None

# Fallback: infer algorithms from current output layout
if not algorithms:
    inferred = set()
    scan_datasets = [d for d in datasets if os.path.isdir(os.path.join(experiment_path, d))]
    for ds_name in scan_datasets:
        model_dir = os.path.join(experiment_path, ds_name, "models", "pickledModels")
        if os.path.isdir(model_dir):
            for fname in os.listdir(model_dir):
                if fname.endswith('.pickle') and '_' in fname:
                    inferred.add(fname.rsplit('_', 1)[0])
        metric_dir = os.path.join(experiment_path, ds_name, "model_evaluation", "metrics_by_cv")
        if os.path.isdir(metric_dir):
            for fname in os.listdir(metric_dir):
                if fname.endswith('.json') and '_CV_' in fname:
                    inferred.add(fname.split('_CV_')[0])
    for abr in sorted(inferred):
        algorithms.append(abr)
        abbrev[abr] = abr
        colors[abr] = None

if requested_algorithms:
    req = set(requested_algorithms)
    filtered = [a for a in algorithms if a in req or abbrev.get(a) in req]
    if filtered:
        algorithms = filtered

palette = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf"]
for idx, key in enumerate(algorithms):
    if colors.get(key) is None:
        colors[key] = palette[idx % len(palette)]

algColors = [colors[k] for k in algorithms]
print("Algorithms Ran: " + str(algorithms))

## Define necessary methods

In [ ]:
def primaryStats(algorithms, original_headers, cv_partitions, full_path, data_name, instance_label, outcome_label, abbrev):
    """Load model evaluation performance from current CSV outputs."""
    metric_dict = {}
    for algorithm in algorithms:
        perf_file = full_path + '/model_evaluation/' + abbrev[algorithm] + '_performance.csv'
        if not os.path.exists(perf_file):
            print('Skipping algorithm with missing performance file: ' + str(algorithm))
            continue
        perf_df = pd.read_csv(perf_file)
        if perf_df.empty:
            print('Skipping algorithm with empty performance file: ' + str(algorithm))
            continue

        # Standardize a few common naming variants
        rename_map = {
            'F1_Score': 'F1 Score',
            'ROC_AUC': 'ROC AUC',
            'PRC_AUC': 'PRC AUC',
            'PRC_APS': 'PRC APS',
            'balanced_accuracy': 'Balanced Accuracy',
            'accuracy': 'Accuracy',
            'f1': 'F1 Score',
            'precision': 'Precision (PPV)',
            'recall': 'Sensitivity (Recall)'
        }
        perf_df = perf_df.rename(columns=rename_map)
        metric_dict[algorithm] = {col: perf_df[col].tolist() for col in perf_df.columns}

    return metric_dict

In [ ]:
def prepFI(algorithms,full_path,abbrev,metric_dict,metric_weight,fi_ranking,fi_weighting):
    """ Organizes and prepares model feature importance data for boxplot and composite feature importance figure generation."""
    def _get_metric_values(metric_data, metric_name):
        candidates = [
            metric_name,
            metric_name.replace(' ', '_'),
            metric_name.replace('_', ' '),
            metric_name.lower(),
            metric_name.lower().replace(' ', '_')
        ]
        for c in candidates:
            if c in metric_data:
                return metric_data[c]
        return None

    fi_df_list = []
    fi_ave_list = []
    ave_metric_list = []
    all_feature_list = []
    used_algorithms = []

    for algorithm in algorithms:
        fi_file = full_path+'/model_evaluation/feature_importance/'+abbrev[algorithm]+'_FI.csv'
        if not os.path.exists(fi_file):
            print('Skipping algorithm with missing FI file: ' + str(algorithm))
            continue
        if algorithm not in metric_dict:
            print('Skipping algorithm with missing metric summary: ' + str(algorithm))
            continue

        temp_df = pd.read_csv(fi_file)
        if temp_df.empty:
            print('Skipping algorithm with empty FI file: ' + str(algorithm))
            continue

        if not all_feature_list:
            all_feature_list = temp_df.columns.tolist()

        fi_df_list.append(temp_df)
        used_algorithms.append(algorithm)

        if fi_ranking == 'mean':
            fi_ave_list.append(temp_df.mean().tolist())
        elif fi_ranking == 'median':
            fi_ave_list.append(temp_df.median().tolist())
        else:
            print("Error: fi_ranking selection not found (must be mean or median)")
            fi_ave_list.append(temp_df.mean().tolist())

        metric_values = _get_metric_values(metric_dict[algorithm], metric_weight)
        if metric_values is None or len(metric_values) == 0:
            avg_metric = 0.5
        else:
            if fi_weighting == 'median':
                avg_metric = median(metric_values)
            else:
                avg_metric = mean(metric_values)
        ave_metric_list.append(avg_metric)

    fi_ave_norm_list = []
    for each in fi_ave_list:
        normList = []
        max_each = max(each) if len(each) > 0 else 0
        for i in range(len(each)):
            if each[i] <= 0 or max_each <= 0:
                normList.append(0)
            else:
                normList.append((each[i]) / max_each)
        fi_ave_norm_list.append(normList)

    if not fi_ave_list:
        return fi_df_list, fi_ave_norm_list, ave_metric_list, all_feature_list, [], [], used_algorithms

    alg_non_zero_FI_list = []
    for each in fi_ave_list:
        temp_non_zero_list = []
        for i in range(len(each)):
            if each[i] > 0.0:
                temp_non_zero_list.append(all_feature_list[i])
        alg_non_zero_FI_list.append(temp_non_zero_list)

    non_zero_union_features = list(alg_non_zero_FI_list[0])
    for j in range(1, len(used_algorithms)):
        non_zero_union_features = list(set(non_zero_union_features) | set(alg_non_zero_FI_list[j]))

    non_zero_union_indexes = []
    for i in non_zero_union_features:
        non_zero_union_indexes.append(all_feature_list.index(i))

    return fi_df_list,fi_ave_norm_list,ave_metric_list,all_feature_list,non_zero_union_features,non_zero_union_indexes,used_algorithms

In [ ]:
def selectForViz(top_model_features,non_zero_union_features,non_zero_union_indexes,algorithms,ave_metric_list,fi_ave_norm_list):
    """ Identify list of top features over all algorithms to visualize (note that best features to vizualize are chosen using algorithm performance weighting and normalization:
    frac plays no useful role here only for viz). All features included if there are fewer than 'top_model_features'. Top features are determined by the sum of performance
    (i.e. balanced accuracy) weighted feature importances over all algorithms."""
    featuresToViz = None
    #Create performance weighted score sum dictionary for all features
    scoreSumDict = {}
    i = 0
    for each in non_zero_union_features:  # for each non-zero feature
        for j in range(len(algorithms)):  # for each algorithm
            # grab target score from each algorithm
            score = fi_ave_norm_list[j][non_zero_union_indexes[i]]
            # multiply score by algorithm performance weight
            weight = ave_metric_list[j]
            if weight <= .5:
                weight = 0
            if not weight == 0:
                weight = (weight - 0.5) / 0.5
            score = score * weight
            #score = score * ave_metric_list[j]
            if not each in scoreSumDict:
                scoreSumDict[each] = score
            else:
                scoreSumDict[each] += score
        i += 1
    # Sort features by decreasing score
    scoreSumDict_features = sorted(scoreSumDict, key=lambda x: scoreSumDict[x], reverse=True)
    if top_model_features == None or not len(non_zero_union_features) > top_model_features:
        featuresToViz = scoreSumDict_features
    else:
        featuresToViz = scoreSumDict_features[0:top_model_features]

    return featuresToViz #list of feature names to vizualize in composite FI plots.

In [ ]:
def getFI_To_Viz_Sorted(featuresToViz,all_feature_list,algorithms,fi_ave_norm_list):
    """ Takes a list of top features names for vizualization, gets their indexes. In every composite FI plot features are ordered the same way
    they are selected for vizualization (i.e. normalized and performance weighted). Because of this feature bars are only perfectly ordered in
    descending order for the normalized + performance weighted composite plot. """
    #Get original feature indexs for selected feature names
    feature_indexToViz = [] #indexes of top features
    for i in featuresToViz:
        feature_indexToViz.append(all_feature_list.index(i))
    # Create list of top feature importance values in original dataset feature order
    top_fi_ave_norm_list = [] #feature importance values of top features for each algorithm (list of lists)
    for i in range(len(algorithms)):
        tempList = []
        for j in feature_indexToViz: #each top feature index
            tempList.append(fi_ave_norm_list[i][j]) #add corresponding FI value
        top_fi_ave_norm_list.append(tempList)
    all_feature_listToViz = featuresToViz
    return top_fi_ave_norm_list,all_feature_listToViz

In [ ]:
def composite_FI_plot(fi_list, algorithms, algColors, all_feature_listToViz, figName,full_path,yLabelText,name_modifier,legend_inside_plot,fi_ranking,fi_weighting,metric_weight):

    algorithms, algColors, fi_list = (list(t) for t in zip(*sorted(zip(algorithms, algColors, fi_list), reverse=True)))
    # Set basic plot properties
    rc('font', weight='bold', size=16)
    # The position of the bars on the x-axis
    r = all_feature_listToViz  # feature names
    # Set width of bars
    bar_width = 0.75
    # Set figure dimensions
    plt.figure(figsize=(24, 12))
    # Plot first algorithm FI scores (lowest) bar
    p1 = plt.bar(r, fi_list[0], color=algColors[0], edgecolor='white', width=bar_width)
    # Automatically calculate space needed to plot next bar on top of the one before it
    bottoms = []  # list of space used by previous
    # algorithms for each feature (so next bar can be placed directly above it)
    bottom = None
    for i in range(len(algorithms) - 1):
        for j in range(i + 1):
            if j == 0:
                bottom = np.array(fi_list[0]).astype('float64')
            else:
                bottom += np.array(fi_list[j]).astype('float64')
        bottoms.append(bottom)
    if not isinstance(bottoms, list):
        bottoms = bottoms.tolist()
    if len(algorithms) > 1:
        # Plot subsequent feature bars for each subsequent algorithm
        ps = [p1[0]]
        for i in range(len(algorithms) - 1):
            p = plt.bar(r, fi_list[i + 1], bottom=bottoms[i], color=algColors[i + 1], edgecolor='white',
                        width=bar_width)
            ps.append(p[0])
        lines = tuple(ps)
    else:
        ps = [p1[0]]
        lines = tuple(ps)
    # Specify axes info and legend
    plt.xticks(np.arange(len(all_feature_listToViz)), all_feature_listToViz, rotation='vertical')
    plt.xlabel("Features (ranked by sum of "+fi_ranking+" feature importance: weighted by "+fi_weighting+" model "+metric_weight.lower()+")", fontsize=20)
    plt.ylabel(yLabelText, fontsize=20)
    algorithms_list, lines_list = (list(t) for t in zip(*sorted(zip(algorithms, lines))))
    if legend_inside_plot:
        plt.legend(lines[::-1], algorithms[::-1],loc="upper right")
    else:
        plt.legend(lines[::-1], algorithms[::-1],loc="upper left", bbox_to_anchor=(1.01,1))
    # Export and/or show plot
    plt.savefig(full_path+'/model_evaluation/feature_importance/Compare_FI_' + figName +name_modifier+ '.png', bbox_inches='tight')
    plt.show()
    

    
    """ Generate composite feature importance plot given list of feature names and associated feature importance scores for each algorithm.
    This is run for different transformations of the normalized feature importance scores. """
    """
    # Set basic plot properites
    rc('font', weight='bold', size=16)
    # The position of the bars on the x-axis
    r = all_feature_listToViz #feature names
    #Set width of bars
    barWidth = 0.75
    #Set figure dimensions
    plt.figure(figsize=(24, 12))
    #Plot first algorithm FI scores (lowest) bar
    p1 = plt.bar(r, fi_list[0], color=algColors[0], edgecolor='white', width=barWidth)
    #Automatically calculate space needed to plot next bar on top of the one before it
    bottoms = [] #list of space used by previous algorithms for each feature (so next bar can be placed directly above it)
    for i in range(len(algorithms) - 1):
        for j in range(i + 1):
            if j == 0:
                bottom = np.array(fi_list[0])
            else:
                bottom += np.array(fi_list[j])
        bottoms.append(bottom)
    if not isinstance(bottoms, list):
        bottoms = bottoms.tolist()
    #Plot subsequent feature bars for each subsequent algorithm
    ps = [p1[0]]
    for i in range(len(algorithms) - 1):
        p = plt.bar(r, fi_list[i + 1], bottom=bottoms[i], color=algColors[i + 1], edgecolor='white', width=barWidth)
        ps.append(p[0])
    lines = tuple(ps)
    # Specify axes info and legend
    plt.xticks(np.arange(len(all_feature_listToViz)), all_feature_listToViz, rotation='vertical')
    plt.xlabel("Feature", fontsize=20)
    plt.ylabel(yLabelText, fontsize=20)
    if legend_inside_plot:
        plt.legend(lines[::-1], algorithms[::-1],loc="upper right")
    else:
        plt.legend(lines[::-1], algorithms[::-1],loc="upper left", bbox_to_anchor=(1.01,1))
    #Export and/or show plot
    plt.savefig(full_path+'/model_evaluation/feature_importance/Compare_FI_' + figName +name_modifier+ '.png', bbox_inches='tight')
    plt.show()
    """

In [ ]:
def fracFI(top_fi_ave_norm_list):
    """ Transforms feature scores so that they sum to 1 over all features for a given algorithm.  This way the normalized and fracionated composit bar plot
    offers equal total bar area for every algorithm. The intuition here is that if an algorithm gives the same FI scores for all top features it won't be
    overly represented in the resulting plot (i.e. all features can have the same maximum feature importance which might lead to the impression that an
    algorithm is working better than it is.) Instead, that maximum 'bar-real-estate' has to be divided by the total number of features. Notably, this
    transformation has the potential to alter total algorithm FI bar height ranking of features. """
    fracLists = []
    for each in top_fi_ave_norm_list: #each algorithm
        fracList = []
        for i in range(len(each)): #each feature
            if sum(each) == 0: #check that all feature scores are not zero to avoid zero division error
                fracList.append(0)
            else:
                fracList.append((each[i] / (sum(each))))
        fracLists.append(fracList)
    return fracLists

In [ ]:
def weightFI(ave_metric_list,top_fi_ave_norm_list):
    """ Weights the feature importance scores by algorithm performance (intuitive because when interpreting feature importances we want to place more weight on better performing algorithms) """
    # Prepare weights
    weights = []
    # replace all balanced accuraces <=.5 with 0 (i.e. these are no better than random chance)
    for i in range(len(ave_metric_list)):
        if ave_metric_list[i] <= .5:
            ave_metric_list[i] = 0
    # normalize balanced accuracies
    for i in range(len(ave_metric_list)):
        if ave_metric_list[i] == 0:
            weights.append(0)
        else:
            weights.append((ave_metric_list[i] - 0.5) / 0.5)
    # Weight normalized feature importances
    weightedLists = []
    for i in range(len(top_fi_ave_norm_list)): #each algorithm
        weightList = np.multiply(weights[i], top_fi_ave_norm_list[i]).tolist()
        weightedLists.append(weightList)
    return weightedLists,weights

In [ ]:
def weightFracFI(fracLists,weights):
    """ Weight normalized and fractionated feature importances. """
    weightedFracLists = []
    for i in range(len(fracLists)):
        weightList = np.multiply(weights[i], fracLists[i]).tolist()
        weightedFracLists.append(weightList)
    return weightedFracLists

## Generate composite feature importance plots

In [ ]:
if targetDataName not in (None, 'None'):
    datasets = [d for d in datasets if d == targetDataName]

for each in datasets: #each analyzed dataset to make plots for
    print("---------------------------------------")
    print("Dataset: "+str(each))
    print("---------------------------------------")
    full_path = experiment_path+'/'+each

    original_headers = pd.read_csv(full_path+"/exploratory/ProcessedFeatureNames.csv",sep=',').columns.values.tolist()

    available_algorithms = [a for a in algorithms if os.path.exists(full_path+'/model_evaluation/feature_importance/'+abbrev[a]+'_FI.csv')]
    if not available_algorithms:
        print('No available FI files for this dataset. Skipping.')
        continue

    metric_dict = primaryStats(available_algorithms,original_headers,cv_partitions,full_path,each,instance_label,outcome_label,abbrev)

    fi_df_list,fi_ave_norm_list,ave_metric_list,all_feature_list,non_zero_union_features,non_zero_union_indexes,used_algorithms = prepFI(available_algorithms,full_path,abbrev,metric_dict,metric_weight,fi_ranking,fi_weighting)

    if not used_algorithms or not non_zero_union_features:
        print('No non-zero FI values available for composite plot. Skipping.')
        continue

    used_alg_colors = [colors.get(a, '#1f77b4') for a in used_algorithms]

    featuresToViz = selectForViz(top_model_features,non_zero_union_features,non_zero_union_indexes,used_algorithms,ave_metric_list,fi_ave_norm_list)

    top_fi_ave_norm_list,all_feature_listToViz = getFI_To_Viz_Sorted(featuresToViz,all_feature_list,used_algorithms,fi_ave_norm_list)

    if viz_norm_only:
        composite_FI_plot(top_fi_ave_norm_list, used_algorithms, used_alg_colors, all_feature_listToViz, 'Norm',full_path, 'Normalized Feature Importance',name_modifier,legend_inside_plot,fi_ranking,fi_weighting,metric_weight)

    fracLists = fracFI(top_fi_ave_norm_list)

    if viz_norm_frac:
        composite_FI_plot(fracLists, used_algorithms, used_alg_colors, all_feature_listToViz, 'Norm_Frac',full_path, 'Normalized and Fractioned Feature Importance',name_modifier,legend_inside_plot,fi_ranking,fi_weighting,metric_weight)

    weightedLists,weights = weightFI(ave_metric_list,top_fi_ave_norm_list)

    if viz_norm_weight:
        composite_FI_plot(weightedLists, used_algorithms, used_alg_colors, all_feature_listToViz, 'Norm_Weight',full_path, 'Normalized and Weighted Feature Importance',name_modifier,legend_inside_plot,fi_ranking,fi_weighting,metric_weight)

    weightedFracLists = weightFracFI(fracLists,weights)

    if viz_norm_weight_frac:
        composite_FI_plot(weightedFracLists, used_algorithms, used_alg_colors, all_feature_listToViz, 'Norm_Frac_Weight',full_path, 'Normalized, Fractioned, and Weighted Feature Importance',name_modifier,legend_inside_plot,fi_ranking,fi_weighting,metric_weight)